# Function Approximation with Noisy Dynamic System
The system is governed by:
$$x_i^5 = 0.1 x_i^0 x_i^1 + 0.5 \sin(x_i^2 x_i^3) + \sin(x_i^4) + \mu_i$$

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
torch.manual_seed(42)
np.random.seed(42)

def generate_data(n_samples, noise_std):
    X = []
    y = []
    
    for _ in range(n_samples):
        x0 = np.random.uniform(-1, 1)
        x1 = np.random.uniform(-1, 1)
        x2 = np.random.uniform(-1, 1)
        x3 = np.random.uniform(-1, 1)
        x4 = np.random.uniform(-1, 1)
        
        mu = np.random.normal(0, noise_std)
        x5 = 0.1 * x0 * x1 + 0.5 * np.sin(x2 * x3) + np.sin(x4) + mu
        
        X.append([x0, x1, x2, x3, x4])
        y.append(x5)
    
    return torch.FloatTensor(np.array(X)), torch.FloatTensor(np.array(y))

noise_std = 0.05

print(f"Generating data with noise level σ={noise_std}")
X_train, y_train = generate_data(n_samples=1000, noise_std=noise_std)
X_test, y_test = generate_data(n_samples=200, noise_std=noise_std)

X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Test y - Mean: {y_test.mean():.4f}, Std: {y_test.std():.4f}")

Generating data with noise level σ=0.05
Train: torch.Size([1000, 5]), Test: torch.Size([200, 5])
Test y - Mean: -0.0524, Std: 0.5199


In [3]:
torch.manual_seed(42)
np.random.seed(42)

kan_model = FastKAN(
    [5, 12, 8, 1],
    grid_min=-1,
    grid_max=1,
    num_grids=8,
    use_base_update=False,
    use_layernorm=False,
).to(device)
kan_optimizer = torch.optim.Adam(kan_model.parameters(), lr=0.001)

In [4]:
epochs = 100
patience = 30
best_loss = float('inf')
patience_counter = 0

print("Training KAN [5, 12, 1]...")
for epoch in range(epochs):
    kan_model.train()
    kan_optimizer.zero_grad()
    pred = kan_model(X_train)
    loss = torch.nn.functional.mse_loss(pred, y_train.unsqueeze(1))
    loss.backward()
    kan_optimizer.step()
    
    kan_model.eval()
    with torch.no_grad():
        test_pred = kan_model(X_test)
        test_loss = torch.nn.functional.mse_loss(test_pred, y_test.unsqueeze(1))
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Test MSE: {test_loss.item():.6f}")
    
    if test_loss < best_loss:
        best_loss = test_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

kan_model.eval()
with torch.no_grad():
    kan_test_pred = kan_model(X_test).squeeze(1)
    kan_mse = mean_squared_error(y_test.cpu().numpy(), kan_test_pred.cpu().numpy())
    kan_r2 = r2_score(y_test.cpu().numpy(), kan_test_pred.cpu().numpy())

print(f"\nKAN Test MSE: {kan_mse:.6f}")
print(f"KAN Test R2: {kan_r2:.4f}")

Training KAN [5, 12, 1]...
Epoch 0, Train MSE: 0.390731, Test MSE: 0.313796
Epoch 20, Train MSE: 0.139098, Test MSE: 0.109560
Epoch 40, Train MSE: 0.063011, Test MSE: 0.054937
Epoch 60, Train MSE: 0.038357, Test MSE: 0.037862
Epoch 80, Train MSE: 0.023988, Test MSE: 0.028999

KAN Test MSE: 0.024705
KAN Test R2: 0.9081


In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

mlp_model = MLP(5, 64, 1).to(device)
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

epochs = 100
patience = 30
best_loss = float('inf')
patience_counter = 0

print("Training XNet [5, 64, 1] (MLP)...")
for epoch in range(epochs):
    mlp_model.train()
    mlp_optimizer.zero_grad()
    mlp_pred = mlp_model(X_train)
    loss = torch.nn.functional.mse_loss(mlp_pred, y_train.unsqueeze(1))
    loss.backward()
    mlp_optimizer.step()
    
    mlp_model.eval()
    with torch.no_grad():
        test_pred = mlp_model(X_test)
        test_loss = torch.nn.functional.mse_loss(test_pred, y_test.unsqueeze(1))
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Test MSE: {test_loss.item():.6f}")
    
    if test_loss < best_loss:
        best_loss = test_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

mlp_model.eval()
with torch.no_grad():
    mlp_test_pred = mlp_model(X_test).squeeze(1)
    mlp_mse = mean_squared_error(y_test.cpu().numpy(), mlp_test_pred.cpu().numpy())
    mlp_r2 = r2_score(y_test.cpu().numpy(), mlp_test_pred.cpu().numpy())

print(f"\nXNet (MLP) Test MSE: {mlp_mse:.6f}")
print(f"XNet (MLP) Test R2: {mlp_r2:.4f}")

Training XNet [5, 32, 1] (MLP)...
Epoch 0, Train MSE: 0.404065, Test MSE: 0.408278
Epoch 20, Train MSE: 0.269919, Test MSE: 0.257094
Epoch 40, Train MSE: 0.179389, Test MSE: 0.166204
Epoch 60, Train MSE: 0.110176, Test MSE: 0.100861
Epoch 80, Train MSE: 0.059494, Test MSE: 0.054607

XNet (MLP) Test MSE: 0.028812
XNet (MLP) Test R2: 0.8928


In [6]:
print("\nAnalyzing KAN predictions...")
individual_losses = []
predictions = []

kan_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_data = X_test[i]
        ground_truth = y_test[i]
        prediction = kan_model(input_data.unsqueeze(0)).squeeze()
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)
mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3]
highest_indices = sorted_indices[-3:]
mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing KAN predictions...
Mean Loss: 0.024705
Min Loss: 0.000004
Max Loss: 0.154586


In [7]:
print("\nAnalyzing MLP predictions...")
mlp_individual_losses = []
mlp_predictions = []

mlp_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)
mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3]
mlp_highest_indices = mlp_sorted_indices[-3:]
mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing MLP predictions...
Mean Loss: 0.028812
Min Loss: 0.000000
Max Loss: 0.224669


In [8]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        input_vector = X_test[idx].cpu().numpy()
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (x0,x1,x2,x3,x4)": f"({input_vector[0]:.3f}, {input_vector[1]:.3f}, {input_vector[2]:.3f}, {input_vector[3]:.3f}, {input_vector[4]:.3f})",
            "True Value": f"{y_test[idx].item():.6f}",
            "Predicted": f"{predictions[idx].item():.6f}",
            "Loss": f"{individual_losses[idx]:.8f}"
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis

,Category,Index,"Input (x0,x1,x2,x3,x4)",True Value,Predicted,Loss
0,Lowest,110,"(0.505, -0.052, -0.106, -0.986, 0.639)",0.676939,0.678938,0.00000400
1,Lowest,115,"(0.670, 0.685, -0.394, -0.141, -0.059)",0.018188,0.020314,0.00000452
2,Lowest,14,"(-0.079, -0.427, -0.505, 0.290, 0.302)",0.212075,0.217243,0.00002671
3,Highest,89,"(-0.738, -0.529, 0.865, 0.719, 0.420)",0.809810,0.453947,0.12663887
4,Highest,90,"(-0.036, 0.041, 0.950, 0.932, -0.569)",-0.178451,-0.552475,0.13989364
5,Highest,199,"(0.095, 0.281, -0.481, -0.831, -0.761)",-0.403168,-0.796342,0.15458575
6,Mean,162,"(0.090, -0.433, 0.897, 0.685, 0.445)",0.652289,0.492079,0.02566719
7,Mean,197,"(0.641, -0.430, -0.522, 0.574, -0.887)",-0.995185,-0.842178,0.02341110
8,Mean,107,"(-0.742, 0.895, -0.050, -0.365, 0.533)",0.407339,0.558836,0.02295120


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        input_vector = X_test[idx].cpu().numpy()
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (x0,x1,x2,x3,x4)": f"({input_vector[0]:.3f}, {input_vector[1]:.3f}, {input_vector[2]:.3f}, {input_vector[3]:.3f}, {input_vector[4]:.3f})",
            "True Value": f"{y_test[idx].item():.6f}",
            "Predicted": f"{mlp_predictions[idx].item():.6f}",
            "Loss": f"{mlp_individual_losses[idx]:.8f}"
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis

,Category,Index,"Input (x0,x1,x2,x3,x4)",True Value,Predicted,Loss
0,Lowest,69,"(-0.855, 0.580, -0.564, 0.354, 0.606)",0.317103,0.317454,0.00000012
1,Lowest,4,"(-0.489, -0.271, 0.687, -0.610, 0.654)",0.494247,0.493176,0.00000115
2,Lowest,145,"(-0.181, 0.560, 0.542, -0.544, 0.844)",0.592539,0.593675,0.00000129
3,Highest,83,"(-0.425, 0.474, -0.093, 0.979, -0.891)",-0.903262,-0.445976,0.20911017
4,Highest,77,"(-0.992, -0.524, 0.877, -0.658, -0.774)",-0.914713,-0.453845,0.21239892
5,Highest,109,"(0.527, -0.727, -0.543, 0.765, -0.960)",-1.088346,-0.614353,0.22466931
6,Mean,180,"(-0.273, -0.566, 0.339, 0.941, -0.649)",-0.504465,-0.335173,0.02865990
7,Mean,137,"(-0.078, 0.343, -0.676, -0.027, 0.477)",0.474354,0.308134,0.02762893
8,Mean,50,"(-0.227, -0.160, 0.124, -0.988, -0.843)",-0.781233,-0.615308,0.02753088


In [12]:
mlp_save_path = "model_pkls/funcnoise_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 5,
        'hidden_dims': [64],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/funcnoise_mlp_model.pkl


In [13]:
kan_save_path = "model_pkls/funcnoise_kan_model.pkl"
torch.save({
    'model_state_dict': kan_model.state_dict(),
    'config': {
        'layers_hidden': [5, 12, 8, 1],
        'grid_min': -1,
        'grid_max': 1,
        'num_grids': 8,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/funcnoise_kan_model.pkl
